In [ ]:
import sys
from pathlib import Path
import os
import mne
import numpy as np
import matplotlib.pyplot as plt


SUBJECT = "16_subj"
SESSION_LABEL = "C"      # Session: C or S
USE_FSAVERAGE = True    # True = fsaverage, False = Native MRI


# Standard params
if USE_FSAVERAGE:
    TEMPLATE = "fsaverage"
    ATLAS = "Destrieux"
    ATLAS_PARC = "aparc.a2009s"
else:
    TEMPLATE = SUBJECT
    ATLAS = "Destrieux"
    ATLAS_PARC = "aparc.a2009s"

# Paths (auto-derived)
MEG_ROOT = Path(r"E:\Develop\MEG")

SUBJECT_MAPPING = {
    "8_subj": MEG_ROOT / "8_subj",
    "1_subj":    MEG_ROOT / "1_subj",
    "6_subj":  MEG_ROOT / "6_subj",
    "9_subj": MEG_ROOT / "9_subj",
    "12_subj":    MEG_ROOT / "12_subj",
    "2_subj": MEG_ROOT / "2_subj",
    "11_subj": MEG_ROOT / "11_subj",
    "7_subj": MEG_ROOT / "7_subj",
    "10_subj": MEG_ROOT / "10_subj",
    "15_subj": MEG_ROOT / "15_subj",
    "16_subj": MEG_ROOT / "16_subj",
}

if SUBJECT not in SUBJECT_MAPPING:
    raise ValueError(f"Subject '{SUBJECT}' not found! Available: {list(SUBJECT_MAPPING.keys())}")

# Base paths
SUBJECT_BASE = SUBJECT_MAPPING[SUBJECT]
DERIV_BASE = SUBJECT_BASE / SESSION_LABEL / "derivatives"
DERIV_NATIVE = DERIV_BASE / "NativeMRI"    # ← inverse lives here
DERIV_FSAVERAGE = DERIV_BASE / "fsaverage"  # ← output goes here

# Select paths based on mode
if USE_FSAVERAGE:
    DERIV_OUTPUT = DERIV_FSAVERAGE
    fsaverage_subject = 'fsaverage'
    # Use LOCAL fsaverage (do not download from internet!)
    FSAVERAGE_ROOT = MEG_ROOT / "freesurfer_subjects"
    subjects_dir = str(FSAVERAGE_ROOT)  # MNE looks for 'fsaverage' inside this dir

    # Verify local fsaverage exists
    local_fsaverage = FSAVERAGE_ROOT / "fsaverage"
    if not local_fsaverage.exists():
        raise ValueError(f"Local fsaverage not found at: {local_fsaverage}\n"
                         f"Expected structure: {FSAVERAGE_ROOT}/fsaverage/")
    print(f"✓ Using LOCAL fsaverage: {local_fsaverage}")
else:
    DERIV_OUTPUT = DERIV_NATIVE
    fsaverage_subject = SUBJECT
    # Native MRI: find FreeSurfer directory (may differ from subject name!)
    freesurfer_dir = None
    print(f"\n[DEBUG] Searching for FreeSurfer directory in: {SUBJECT_BASE}")
    print(f"[DEBUG] Contents of {SUBJECT_BASE.name}:")
    for item in SUBJECT_BASE.iterdir():
        print(f"  - {item.name} (is_dir={item.is_dir()})")
        if item.is_dir():
            has_label = (item / "label").exists()
            has_bem = (item / "bem").exists()
            print(f"      has 'label': {has_label}, has 'bem': {has_bem}")
            if has_label or has_bem:
                freesurfer_dir = item
                print(f"      ✓ SELECTED as FreeSurfer dir")

    if freesurfer_dir is None:
        raise ValueError(
            f"Cannot find FreeSurfer directory in {SUBJECT_BASE}\n"
            f"Expected: subdir with 'label' folder (e.g., 'Ryzhik' for subject '12ryzhik')"
        )

    FREESURFER_NAME = freesurfer_dir.name
    subjects_dir = str(SUBJECT_BASE)  # MNE will look for FREESURFER_NAME subfolder inside
    print(f"\n✓ Found FreeSurfer directory: {FREESURFER_NAME}")
    print(f"✓ subjects_dir for MNE: {subjects_dir}")

# Try both epoch filename variants
_epochs_candidates = [
    DERIV_BASE / f"epochs_combined_{SESSION_LABEL}-epo.fif",
    DERIV_BASE / f"epochs_with_master_{SESSION_LABEL}-epo.fif",
]
EPOCHS_FILE = next((p for p in _epochs_candidates if p.exists()), _epochs_candidates[0])

# Inverse operator path depends on mode
# BIDS-style: sub-{SUBJECT}_ses-{SESSION}_desc-{mode}_inv.fif
if USE_FSAVERAGE:
    INV_SEARCH_DIR = DERIV_FSAVERAGE  # ← fsaverage inverse folder
    INV_PATTERN = f"sub-{SUBJECT}_ses-{SESSION_LABEL}_desc-*_inv.fif"  # BIDS-style
else:
    INV_SEARCH_DIR = DERIV_NATIVE  # ← NativeMRI inverse folder
    INV_PATTERN = f"sub-{SUBJECT}_ses-{SESSION_LABEL}_desc-*_inv.fif"  # BIDS-style

# Output file (for Step 5)
OUTPUT_NPZ = DERIV_OUTPUT / f"roi_tc_{ATLAS}_{SESSION_LABEL}.npz"

# Verify
print("="*70)
print(f"4.2 EXTRACT ROI TIMECOURSES — {SUBJECT} / {SESSION_LABEL}")
print("="*70)
print(f"\n[PATHS]")
print(f"  Subject base: {SUBJECT_BASE}")
print(f"  Output:       {DERIV_OUTPUT}")

print(f"\n[INPUT FILES]")
print(f"  ⏳ {EPOCHS_FILE.name}  (loaded in Cell 2)")
print(f"  ⏳ Inverse files from: {INV_SEARCH_DIR.name}")

print(f"\n[OUTPUT]")
print(f"  → {OUTPUT_NPZ.name}  ★ KEY OUTPUT for Step 5.1")
print("="*70)


# ROI DEFINITIONS: canonical vmPFC and dlPFC parcels
_repo_root = Path(r"E:\Develop\MEG\Preprocessing_Pipeline")
import sys
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from pipeline_utils.roi_definitions import (
    VMPFC_LABELS, DLPFC_LABELS, get_roi_labels, VMPFC_PARCELS, DLPFC_PARCELS
)
from pipeline_utils.anatomy_layer import get_anatomy




print(f"\n[ROI DEFINITIONS]")
print(f"  (both hemispheres = {len(VMPFC_LABELS)} vmPFC + {len(DLPFC_LABELS)} dlPFC labels)")

print(f"  vmPFC parcels ({len(VMPFC_PARCELS)}): {VMPFC_PARCELS}")
print(f"  dlPFC parcels ({len(DLPFC_PARCELS)}): {DLPFC_PARCELS}")


In [ ]:
# IMPORTANT: load labels with the correct subject parameter
print(f"[LOAD] Loading {ATLAS} atlas...")
print(f"  subjects_dir: {subjects_dir}")
print(f"  subject: {fsaverage_subject}\n")

try:
    # Native MRI: subject = FreeSurfer folder name inside subjects_dir
    # fsaverage: subject = 'fsaverage'
    if USE_FSAVERAGE:
        subject_for_labels = fsaverage_subject  # 'fsaverage'
    else:
        # Native MRI: use FreeSurfer folder name (may differ from subject name!)
        subject_for_labels = FREESURFER_NAME

    labels = mne.read_labels_from_annot(
        subject=subject_for_labels,
        parc="aparc.a2009s",        # Destrieux atlas
        subjects_dir=subjects_dir,
        verbose=False
    )
    label_names = [label.name for label in labels]
    n_rois = len(labels)

    print(f"  ✓ Loaded {n_rois} labels from {ATLAS}")
    print(f"  ✓ First 5 labels: {label_names[:5]}")

    # Check subject in labels
    print(f"\n[CHECK] Label subjects:")
    print(f"  Label[0].subject = '{labels[0].subject}'")

    # IMPORTANT: ensure all labels have the correct subject
    if USE_FSAVERAGE:
        correct_subject = fsaverage_subject  # 'fsaverage'
    else:
        correct_subject = FREESURFER_NAME  # FreeSurfer dir name (now = SUBJECT)

    for label in labels:
        if label.subject != correct_subject:
            label.subject = correct_subject

    print(f"  ✓ All labels set to subject='{correct_subject}'\n")

except Exception as e:
    print(f" Failed to load labels: {str(e)}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


In [ ]:
# CANONICAL vmPFC / dlPFC LABEL SETS
# Uses fixed parcels from pipeline_utils/roi_definitions.py
# These are the ONLY label sets used for the vmPFC vs dlPFC
# dissociation analysis — identical across all subjects/sessions.

print(f"[ROI] Extracting canonical vmPFC and dlPFC label sets...\n")

vmpfc_labels = get_roi_labels(labels, VMPFC_LABELS, strict=False)
dlpfc_labels = get_roi_labels(labels, DLPFC_LABELS, strict=False)

vmpfc_idx = [label_names.index(l.name) for l in vmpfc_labels if l.name in label_names]
dlpfc_idx = [label_names.index(l.name) for l in dlpfc_labels if l.name in label_names]

print(f"  vmPFC labels found ({len(vmpfc_labels)}):")
for lb in vmpfc_labels:
    print(f"    [{label_names.index(lb.name):3d}] {lb.name}")

print(f"\n  dlPFC labels found ({len(dlpfc_labels)}):")
for lb in dlpfc_labels:
    print(f"    [{label_names.index(lb.name):3d}] {lb.name}")

missing_vmpfc = set(VMPFC_LABELS) - set(label_names)
missing_dlpfc = set(DLPFC_LABELS) - set(label_names)
if missing_vmpfc:
    print(f"\n Missing vmPFC labels (not in atlas): {missing_vmpfc}")
if missing_dlpfc:
    print(f" Missing dlPFC labels (not in atlas): {missing_dlpfc}")
if not missing_vmpfc and not missing_dlpfc:
    print(f"\n  ✓ All canonical ROI labels found in atlas")


In [ ]:
# LOAD EPOCHS AND INVERSE OPERATOR


print("="*80)
print("LOADING EPOCHS AND INVERSE OPERATOR")
print("="*80 + "\n")

# Load epochs
print(f"[LOAD] Loading epochs from: {EPOCHS_FILE.name}")
if not EPOCHS_FILE.exists():
    raise FileNotFoundError(f" Epochs file not found: {EPOCHS_FILE}")

epochs = mne.read_epochs(str(EPOCHS_FILE), verbose=False)
n_trials = len(epochs)
print(f"  ✓ Loaded {n_trials} trials")
print(f"  ✓ Times: {epochs.times[0]:.3f} to {epochs.times[-1]:.3f} s")
print(f"  ✓ Channels: {epochs.info['nchan']}")

# Find inverse file
print(f"\n[SEARCH] Looking for inverse operator in: {INV_SEARCH_DIR.name}")
print(f"  Pattern: {INV_PATTERN}")

inv_files = list(INV_SEARCH_DIR.glob(INV_PATTERN))
if not inv_files:
    raise FileNotFoundError(
        f"No inverse files found matching '{INV_PATTERN}' in {INV_SEARCH_DIR}"
    )

inv_file = inv_files[0]
print(f"  ✓ Found: {inv_file.name}")

# Load inverse
inverse_operator = mne.minimum_norm.read_inverse_operator(str(inv_file))
print(f"  ✓ Loaded inverse operator")

# MNE params
lambda2 = 1.0 / 9.0  # regularization
method = 'MNE'  # or 'dSPM', 'sLORETA'

print(f"\n[PARAMS]")
print(f"  Method: {method}")
print(f"  Lambda²: {lambda2:.4f}")
print(f"  SNR: {1.0 / lambda2:.1f}\n")


In [ ]:
# PER-TRIAL SOURCE INVERSIONS AND ROI TIMECOURSE EXTRACTION

print("="*80)
print("PER-TRIAL SOURCE INVERSIONS")
print("="*80 + "\n")

# RESET containers
roi_tc_all_trials = []
trial_mask = []

# Compatibility check before starting
print(f"[COMPATIBILITY CHECK]")
print(f"  subject: '{fsaverage_subject}'")
print(f"  labels[0].subject: '{labels[0].subject}'")
print(f"  inverse src subject: '{inverse_operator['src'][0].get('subject_his_id', 'N/A')}'")

# Source space from inverse operator
src = inverse_operator['src']

# IMPORTANT: update subject in source space to match labels and stc
# Required for extract_label_time_course
if USE_FSAVERAGE:
    correct_src_subject = fsaverage_subject  # 'fsaverage'
else:
    correct_src_subject = FREESURFER_NAME

for src_part in src:
    src_part['subject_his_id'] = correct_src_subject

print(f"  Source space: {src[0]['nuse']} + {src[1]['nuse']} = {src[0]['nuse'] + src[1]['nuse']} vertices")
print(f"  Source space subject updated to: '{correct_src_subject}'\n")

# NOTE: do NOT filter channels! apply_inverse handles extra channels automatically
# (3 extra channels: ECG063, EOG061, EOG062 will be ignored)
print(f"[CHANNEL CHECK]")
print(f"  Epochs channels: {epochs.info['nchan']}")
print(f"  Inverse channels: {inverse_operator['info']['nchan']}")
print(f"  Extra channels: ECG063, EOG061, EOG062 (will be automatically handled)\n")

# Process each trial
for trial_idx in range(n_trials):
    print(f"  Processing trial : {trial_idx + 1}", flush=True)

    try:
        # Take one trial WITHOUT channel filtering
        epoch_single = epochs[trial_idx:trial_idx+1]

        # Debug on first trial BEFORE inverse
        if trial_idx == 0:
            print(f"    [BEFORE INVERSE]")
            print(f"      Epoch data shape: {epoch_single.get_data().shape}")
            print(f"      Epoch data range: [{epoch_single.get_data().min():.6e}, {epoch_single.get_data().max():.6e}]")
            print(f"      Epoch nchan: {epoch_single.info['nchan']}")
            print(f"      Inverse nchan: {inverse_operator['info']['nchan']}")

        # Apply inverse - MNE automatically selects channels
        stc = mne.minimum_norm.apply_inverse_epochs(
            epochs=epoch_single,
            inverse_operator=inverse_operator,
            lambda2=lambda2,
            method=method,
            verbose=False,
            pick_ori=None
        )[0]

        # Debug info on first trial
        if trial_idx == 0:
            print(f"    [DEBUG] STC subject: '{stc.subject}'")
            print(f"    [DEBUG] STC vertices: lh={len(stc.vertices[0])}, rh={len(stc.vertices[1])}")
            print(f"    [DEBUG] STC data shape: {stc.data.shape}")
            print(f"    [DEBUG] STC data range: [{stc.data.min():.6e}, {stc.data.max():.6e}]")
            print(f"    [DEBUG] STC mean: {stc.data.mean():.6e}, std: {stc.data.std():.6e}")
            print(f"    [DEBUG] Non-zero STC values: {np.count_nonzero(stc.data)}/{stc.data.size}")

        # CRITICAL FIX: STC and labels must share the same subject
        # Native MRI: use FREESURFER_NAME (FreeSurfer folder name)
        # fsaverage: use fsaverage_subject
        if USE_FSAVERAGE:
            stc.subject = fsaverage_subject  # 'fsaverage'
        else:
            stc.subject = FREESURFER_NAME  # FreeSurfer folder name (now = SUBJECT)

        if trial_idx == 0:
            print(f"    [FIX] STC subject set to: '{stc.subject}'")

        # Extract timecourses for all ROIs in one call
        # More efficient and reliable than one label at a time
        try:
            tc_all = mne.extract_label_time_course(
                stcs=stc,
                labels=labels,
                src=src,
                mode='mean',
                verbose=False
            )
            # tc_all shape: (n_labels, n_times)
            roi_tc_trial = tc_all

            if trial_idx == 0:
                print(f"    [SUCCESS] Extracted {roi_tc_trial.shape[0]} ROIs, {roi_tc_trial.shape[1]} timepoints")
                print(f"    [DATA] Range: [{roi_tc_trial.min():.6e}, {roi_tc_trial.max():.6e}]")
                print(f"    [DATA] Mean: {roi_tc_trial.mean():.6e}, Std: {roi_tc_trial.std():.6e}")
                nonzero_rois = np.sum(np.abs(roi_tc_trial).max(axis=1) > 0)
                print(f"    [DATA] Non-zero ROIs: {nonzero_rois}/{roi_tc_trial.shape[0]}")
                print(f"    [DATA] Non-zero values in ROI TC: {np.count_nonzero(roi_tc_trial)}/{roi_tc_trial.size}")

        except Exception as extract_err:
            print(f"    ERROR: extract_label_time_course failed: {str(extract_err)[:100]}")
            if trial_idx == 0:
                import traceback
                traceback.print_exc()
            trial_mask.append(False)
            continue

        # Check: if ALL values are exactly zero (not just small), something is wrong
        # Use strict criterion: np.all(roi_tc_trial == 0) rather than np.allclose
        if np.all(roi_tc_trial == 0):
            if trial_idx < 5:  # only print for first few trials
                print(f"    WARNING Trial {trial_idx}: ALL VALUES ARE EXACTLY ZERO!")
            trial_mask.append(False)  # only truly zero trials flagged as bad
        else:
            trial_mask.append(True)

        roi_tc_all_trials.append(roi_tc_trial)

    except Exception as e:
        print(f"    ERROR Trial {trial_idx} failed: {str(e)[:80]}")
        trial_mask.append(False)

print(f"\n✓ Successfully processed {sum(trial_mask)}/{n_trials} trials")
print(f"  Valid trials in list: {len(roi_tc_all_trials)}\n")

# Stack into a single matrix
if len(roi_tc_all_trials) > 0:
    roi_tc_final = np.stack(roi_tc_all_trials, axis=0)  # (n_valid_trials, n_rois, n_times)
else:
    print("ERROR: NO VALID TRIALS!")
    sys.exit(1)

trial_mask = np.array(trial_mask)

# Diagnostics
print(f"[MATRIX] Final ROI timecourses shape: {roi_tc_final.shape}")
print(f"  (n_trials × n_rois × n_times) = ({roi_tc_final.shape[0]} × {roi_tc_final.shape[1]} × {roi_tc_final.shape[2]})")
print(f"  Data range: [{roi_tc_final.min():.6e}, {roi_tc_final.max():.6e}]")
print(f"  Mean: {roi_tc_final.mean():.6e}, Std: {roi_tc_final.std():.6e}\n")

# Per-ROI diagnostics
print(f"[DIAGNOSE] Per-ROI data ranges:")
roi_maxs = roi_tc_final.max(axis=(0, 2))

zero_roi_indices = np.where(roi_maxs == 0)[0]
nonzero_roi_indices = np.where(roi_maxs > 0)[0]

print(f"  • Total ROIs: {roi_tc_final.shape[1]}")
print(f"  • ROIs with data (max > 0): {len(nonzero_roi_indices)}")
print(f"  • ROIs with all zeros: {len(zero_roi_indices)}")

if len(zero_roi_indices) > 0 and len(zero_roi_indices) <= 10:
    print(f"\n  Zero-valued ROI names:")
    for idx in zero_roi_indices:
        print(f"    [{idx}] {label_names[idx]}")

# CRITICAL CHECK: DLPFC/VMPFC ROIs
dlpfc_patterns = ["G_front_sup", "G_front_middle", "S_front_middle", "G_precentral"]
vmpfc_patterns = ["G_cingul", "G_and_S_cingul", "G_front_inf-Orbital"]

print(f"\n[CRITICAL] Checking DLPFC/VMPFC ROIs:")
for pattern in dlpfc_patterns + vmpfc_patterns:
    for idx, name in enumerate(label_names):
        if pattern in name:
            max_val = roi_maxs[idx]
            status = "OK" if max_val > 0 else "ZERO"
            print(f"  [{idx}] {name:45s} max={max_val:.6e} {status}")
            break
print()


In [ ]:
# SAVE NPZ

print("="*80)
print("SAVE NPZ")
print("="*80 + "\n")


roi_tc_save = roi_tc_final  # (n_valid_trials, n_rois, n_times)
trial_mask_save = trial_mask

print(f"[SAVE] Saving ROI timecourses to NPZ...")
print(f"  File: {OUTPUT_NPZ}")
print(f"  Shape: {roi_tc_save.shape}")
print(f"  Valid trials: {roi_tc_save.shape[0]}\n")

# Ensure output directory exists
OUTPUT_NPZ.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    str(OUTPUT_NPZ),
    data=roi_tc_save, # (n_valid_trials, n_rois, n_times)
    label_names=np.array(label_names, dtype=object),
    times=epochs.times,
    trial_mask=trial_mask_save,
)

print(f"✓ Saved to {OUTPUT_NPZ.name}")
print(f"\n  NPZ contents:")
print(f"  • data: {roi_tc_save.shape} (trials × ROIs × times)")
print(f"  • label_names: {len(label_names)} names")
print(f"  • times: {len(epochs.times)} time points")
print(f"  • trial_mask: {trial_mask_save.sum()} / {len(trial_mask_save)} valid trials")

# FINAL VERIFICATION
print(f"\n[VERIFY] Data integrity check:")
print(f"  • Min value: {roi_tc_save.min():.6e}")
print(f"  • Max value: {roi_tc_save.max():.6e}")
print(f"  • Mean: {roi_tc_save.mean():.6e}")
print(f"  • Non-zero ROIs: {(roi_tc_save.max(axis=(0,2)) > 0).sum()}/{roi_tc_save.shape[1]}")

if roi_tc_save.max() == 0:
    print("\n WARNING: ALL DATA IS ZERO! Something went wrong with label extraction!")
else:
    print(f"\n{'='*80}")
    print(f"✓✓✓ PER-TRIAL ROI TIMECOURSES SUCCESSFULLY CREATED! ✓✓✓")
    print(f"{'='*80}")
    print(f"\nReady for GLM analysis in 5.1!")


In [ ]:
# QUICK RESTART — loads everything from saved files.
# Run this cell after kernel restart instead of slow
# per-trial dSPM inversions (Cells 5–6).
#
# Requires: Cell 1 (config) already run.

print("="*70)
print("QUICK RESTART — loading from saved files")
print("="*70)

# 1. Epochs (needed for times, info, n_trials)
print(f"\n[1] Loading epochs...")
epochs = mne.read_epochs(str(EPOCHS_FILE), verbose=False)
n_trials = len(epochs)
print(f"  ✓ {n_trials} trials, {len(epochs.times)} timepoints")

# 2. Atlas labels
print(f"\n[2] Loading Destrieux labels...")
subject_for_labels = fsaverage_subject if USE_FSAVERAGE else FREESURFER_NAME
labels = mne.read_labels_from_annot(
    subject=subject_for_labels, parc="aparc.a2009s",
    subjects_dir=subjects_dir, verbose=False,
)
label_names = [lb.name for lb in labels]
correct_subject = fsaverage_subject if USE_FSAVERAGE else FREESURFER_NAME
for lb in labels:
    lb.subject = correct_subject
print(f"  ✓ {len(labels)} labels loaded")

# 3. vmPFC / dlPFC indices
print(f"\n[3] Computing vmPFC/dlPFC indices...")
vmpfc_labels = get_roi_labels(labels, VMPFC_LABELS, strict=False)
dlpfc_labels = get_roi_labels(labels, DLPFC_LABELS, strict=False)
vmpfc_idx = [label_names.index(l.name) for l in vmpfc_labels if l.name in label_names]
dlpfc_idx = [label_names.index(l.name) for l in dlpfc_labels if l.name in label_names]
print(f"  ✓ vmPFC: {len(vmpfc_idx)} labels, dlPFC: {len(dlpfc_idx)} labels")

# 4. Load previously saved dSPM NPZ (roi_tc_final + trial_mask)
print(f"\n[4] Loading dSPM NPZ: {OUTPUT_NPZ.name}...")
if not OUTPUT_NPZ.exists():
    raise FileNotFoundError(
        f" dSPM NPZ not found: {OUTPUT_NPZ}\n"
        f"Run Cells 4–6 (per-trial inversions) at least once first!"
    )
_npz = np.load(str(OUTPUT_NPZ), allow_pickle=True)
roi_tc_final = _npz["data"]          # (n_valid_trials, n_rois, n_times)
trial_mask   = _npz["trial_mask"]    # (n_trials,) bool
del _npz
print(f"  ✓ roi_tc_final shape: {roi_tc_final.shape}")
print(f"  ✓ trial_mask: {trial_mask.sum()}/{len(trial_mask)} valid trials")

print(f"\n✓ Quick restart done — ready for LCMV-0 and beyond!")


In [ ]:
# STEP LCMV-0: Load prerequisites (fwd + noise_cov)
print(f"[LCMV-0] Loading forward solution and noise covariance...\n")

# Forward solution (saved by notebook 2.2 STEP 8.1)
fwd_mode = "fsaverage" if USE_FSAVERAGE else "subjectMRI"
fwd_path = DERIV_OUTPUT / f"sub-{SUBJECT}_ses-{SESSION_LABEL}_desc-{fwd_mode}_fwd.fif"

if not fwd_path.exists():
    raise FileNotFoundError(
        f"Forward solution not found: {fwd_path}\n"
        f"Run notebook 2.2 (STEP 8.1) first to generate it."
    )
fwd = mne.read_forward_solution(str(fwd_path))
print(f"  ✓ fwd loaded: {fwd_path.name}")

# Noise covariance — prefer empty-room, fallback to baseline
noise_cov = None
noise_cov_source_lcmv = None
for _desc in ("emptyroom", "baseline"):
    _nc_path = DERIV_BASE / f"sub-{SUBJECT}_ses-{SESSION_LABEL}_desc-{_desc}_noise-cov.fif"
    if _nc_path.exists():
        noise_cov = mne.read_cov(str(_nc_path))
        noise_cov_source_lcmv = _desc
        print(f"  ✓ noise_cov loaded ({_desc}): {_nc_path.name}")
        break

if noise_cov is None:
    print(f"  ⚠️  No cached noise_cov found — computing from baseline")
    noise_cov = mne.compute_covariance(epochs, tmin=None, tmax=0,
                                       method="shrinkage", verbose=False)
    noise_cov_source_lcmv = "baseline-computed"


In [ ]:

# STEP LCMV-1: LCMV beamformer ROI time series
# Used as leakage-control / robustness check for dSPM.
# If vmPFC vs dlPFC dissociation survives in LCMV → spatial
# leakage is not driving the effect.
#
#   Data covariance computed from ALL epochs in this session
#   (tmin=0 to end of epoch, both conditions pooled).
#   The SAME covariance → SAME spatial filters are applied to
#   every trial, regardless of condition. This prevents
#   'cherry-picking' of spatial filters per condition.

from pipeline_utils.lcmv_roi import compute_lcmv_roi_timeseries

# LCMV_PARAMS: all parameters fixed here for reproducibility.
LCMV_PARAMS = {
    "method":               "LCMV",
    "mode":                 "mean_flip",          # sign-flip-consistent mean across label vertices
    "data_cov_tmin_s":      0.0,                  # data covariance window start (post-stimulus)
    "data_cov_tmax_s":      None,                 # None = to end of epoch
    "data_cov_note":        "all trials in session pooled, post-stimulus window",
    "reg":                  0.05,                 # Tikhonov regularization (5%)
    "pick_ori":             "max-power",          # scalar beamformer (max-power orientation)
    "weight_norm":          "unit-noise-gain",    # output unit = relative signal change
    "weights_same_per_condition": True,           # ← KEY: one shared covariance → same weights for all conditions
}

print(f"[LCMV] Computing LCMV ROI timeseries...\n")
print(f"  subjects: {SUBJECT}, session: {SESSION_LABEL}")
print(f"  n_trials: {n_trials}  n_labels: {len(labels)}")
print(f"  reg={LCMV_PARAMS['reg']}  pick_ori={LCMV_PARAMS['pick_ori']}  mode={LCMV_PARAMS['mode']}")
print(f"  data_cov window: [{LCMV_PARAMS['data_cov_tmin_s']}, {LCMV_PARAMS['data_cov_tmax_s']}] s  ({LCMV_PARAMS['data_cov_note']})")

roi_tc_lcmv, label_names_lcmv = compute_lcmv_roi_timeseries(
    epochs=epochs,
    fwd=fwd,
    noise_cov=noise_cov,
    labels=labels,
    mode=LCMV_PARAMS["mode"],
    data_cov_tmin=LCMV_PARAMS["data_cov_tmin_s"],
    data_cov_tmax=LCMV_PARAMS["data_cov_tmax_s"],
    reg=LCMV_PARAMS["reg"],
    pick_ori=LCMV_PARAMS["pick_ori"],
)

print(f"\n  LCMV timeseries shape: {roi_tc_lcmv.shape}")
print(f"  (n_trials × n_rois × n_times) = {roi_tc_lcmv.shape}")

# Save LCMV NPZ alongside the dSPM NPZ
lcmv_out = DERIV_OUTPUT / f"roi_tc_LCMV_{SESSION_LABEL}.npz"
np.savez(
    str(lcmv_out),
    data=roi_tc_lcmv,
    label_names=np.array(label_names_lcmv, dtype=object),
    times=epochs.times,
    trial_mask=trial_mask,
    **{k: v for k, v in LCMV_PARAMS.items()
       if isinstance(v, (str, int, float, bool)) and v is not None},
)
print(f"\n  ✓ LCMV NPZ saved: {lcmv_out.name}")
print(f"  ✓ LCMV_PARAMS saved to NPZ (weights_same_per_condition={LCMV_PARAMS['weights_same_per_condition']})")


In [ ]:
# STEP LCMV-2: Leakage control QC
#
# Three fixes over the naive version:
#   1. Correlations computed on trialwise RESIDUALS (after subtracting evoked).
#   2. Effects z-scored before comparison (dSPM and LCMV in same units).
#   3. LCMV sign aligned to dSPM.
#
# ORTH: C^{-1/2} whitening (no amplitude rescaling).
#   Rescaling after orth mathematically guarantees corr → -r (not 0!),
#   making the metric meaningless. Without rescaling: corr(orth_vm, orth_dl) = 0.
#   Why: Cov(W·X) = W·C·W^T = I by construction. Effects z-scored → amplitude irrelevant.

import sys, importlib

lcmv_out = DERIV_OUTPUT / f"roi_tc_LCMV_{SESSION_LABEL}.npz"
if not isinstance(roi_tc_lcmv, np.ndarray):
    if not lcmv_out.exists():
        raise FileNotFoundError(
            f"LCMV NPZ not found: {lcmv_out}\n"
            f"Run LCMV-1 cell at least once first!"
        )
_lcmv_npz = np.load(str(lcmv_out), allow_pickle=True)
roi_tc_lcmv = _lcmv_npz["data"]
del _lcmv_npz
print(f" ✓ Loaded roi_tc_lcmv from NPZ: {roi_tc_lcmv.shape}")
# Flush all cached module variants
for _key in list(sys.modules.keys()):
    if 'leakage_control' in _key:
        del sys.modules[_key]

from pipeline_utils.leakage_control import compute_leakage_qc
import matplotlib.pyplot as plt

print("[LEAKAGE QC] Computing (trialwise residuals / z-scored effects / sign-aligned)...")

leakage_qc = compute_leakage_qc(
    tc_dspm=roi_tc_final,
    tc_lcmv=roi_tc_lcmv,
    vmpfc_indices=vmpfc_idx,
    dlpfc_indices=dlpfc_idx,
    times=epochs.times,
    tmin_effect=0.1,
    tmax_effect=0.5,
)

corr_raw  = leakage_qc["corr_vmpfc_dlpfc_dspm_raw"]
corr_orth = leakage_qc["corr_vmpfc_dlpfc_dspm_orth"]
corr_lcmv = leakage_qc["corr_vmpfc_dlpfc_lcmv"]
corr_ev_raw  = leakage_qc["corr_evoked_dspm_raw"]
corr_ev_orth = leakage_qc["corr_evoked_dspm_orth"]
corr_ev_lcmv = leakage_qc["corr_evoked_lcmv"]
eff_raw   = leakage_qc["effect_dspm_raw"]
eff_orth  = leakage_qc["effect_dspm_orth"]
eff_lcmv  = leakage_qc["effect_lcmv"]
sign_ok   = leakage_qc["effect_sign_consistent"]
shared_pct = leakage_qc["shared_variance_pct_dspm_residuals"]

#  ORTH SANITY CHECK
print(f"\n  [ORTH SANITY] max|raw - orth| signal amplitude:")
print(f"    vmPFC: {leakage_qc['orth_max_diff_vmpfc']:.6e}")
print(f"    dlPFC: {leakage_qc['orth_max_diff_dlpfc']:.6e}")
if leakage_qc['orth_max_diff_vmpfc'] < 1e-15 and leakage_qc['orth_max_diff_dlpfc'] < 1e-15:
    print(f"  WARNING: orth had NO effect — raw and orth are identical!")
else:
    print(f"    ✓ orth changed the signals")

#  SHARED VARIANCE
print(f"\n  [SHARED VARIANCE] r²(vmPFC, dlPFC) on trialwise residuals: {shared_pct:.1f}%")
if shared_pct > 70:
    print(f" High shared variance — spatial leakage may inflate correlation.")
    print(f"    LCMV comparison is primary leakage control.")
elif shared_pct > 30:
    print(f"    Moderate shared variance — orth is meaningful.")
else:
    print(f"    Low shared variance — leakage unlikely.")

#  CORRELATIONS
print(f"\n  vmPFC-dlPFC correlation (trialwise RESIDUALS):")
print(f"    dSPM raw:          {corr_raw:.3f}  (shared var: {shared_pct:.1f}%)")
print(f"    dSPM orth (C⁻½):  {corr_orth:.3f}")
print(f"    LCMV:              {corr_lcmv:.3f}")
print(f"\n  vmPFC-dlPFC correlation (EVOKED avg — reference):")
print(f"    dSPM raw:          {corr_ev_raw:.3f}")
print(f"    dSPM orth (C⁻½):  {corr_ev_orth:.3f}")
print(f"    LCMV:              {corr_ev_lcmv:.3f}")

# EFFECTS
print(f"\n  Effect (vmPFC−dlPFC, z-scored, 0.1–0.5 s) [σ]:")
print(f"    dSPM raw:   {eff_raw:.4f}")
print(f"    dSPM orth:  {eff_orth:.4f}  ← survives orth → not pure leakage")
print(f"    LCMV:       {eff_lcmv:.4f}  ← survives LCMV → not pure leakage")
if abs(eff_raw) < 0.05:
    print(f"Effect near zero — sign consistency metric unreliable at this scale.")
print(f"\n  Sign alignment (LCMV aligned to dSPM):")
print(f"    vmPFC flipped: {leakage_qc['lcmv_vmpfc_sign_flipped']}")
print(f"    dlPFC flipped: {leakage_qc['lcmv_dlpfc_sign_flipped']}")
print(f"  Sign consistent (dSPM vs LCMV): {sign_ok}")

# Diagnostic figure
_vm_sign = -1 if leakage_qc["lcmv_vmpfc_sign_flipped"] else 1
_dl_sign = -1 if leakage_qc["lcmv_dlpfc_sign_flipped"] else 1

dspm_avg  = roi_tc_final.mean(axis=0)
lcmv_avg  = roi_tc_lcmv.mean(axis=0)
dspm_vmpfc = dspm_avg[vmpfc_idx, :].mean(axis=0)
dspm_dlpfc = dspm_avg[dlpfc_idx, :].mean(axis=0)
lcmv_vmpfc = lcmv_avg[vmpfc_idx, :].mean(axis=0) * _vm_sign
lcmv_dlpfc = lcmv_avg[dlpfc_idx, :].mean(axis=0) * _dl_sign

def _zscore_tc(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-15 else x

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (v, d, title) in zip(axes, [
    (_zscore_tc(dspm_vmpfc), _zscore_tc(dspm_dlpfc), f"dSPM (resid r={corr_raw:.2f}, shared {shared_pct:.0f}%)"),
    (_zscore_tc(lcmv_vmpfc), _zscore_tc(lcmv_dlpfc), f"LCMV sign-aligned (r={corr_lcmv:.2f})"),
]):
    ax.plot(epochs.times * 1000, v, label="vmPFC", color="steelblue")
    ax.plot(epochs.times * 1000, d, label="dlPFC", color="tomato")
    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.5)
    ax.axvspan(100, 500, alpha=0.07, color="gold", label="effect window")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("z-score")
    ax.set_title(f"{SUBJECT} | {SESSION_LABEL} | {title}")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle("Leakage control: vmPFC vs dlPFC (z-scored evoked, sign-aligned)", fontsize=11)
fig.tight_layout()
leakage_fig = DERIV_OUTPUT / f"leakage_control_{SESSION_LABEL}.png"
fig.savefig(str(leakage_fig), dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  ✓ Figure: {leakage_fig.name}")


In [ ]:
# This file is the single source of truth for all parameters
# Works without LCMV cells — LCMV fields will be null/None.

import json as _json
from pipeline_utils.roi_definitions import ROI_CONFIG

_lcmv_params   = globals().get("LCMV_PARAMS", None)
_lcmv_noise    = globals().get("noise_cov_source_lcmv", "not_run")
_leakage_qc    = globals().get("leakage_qc", None)
_lcmv_run      = _lcmv_params is not None
_leakage_run   = _leakage_qc is not None

roi_meta = {
    # Identity
    "subject":      SUBJECT,
    "session":      SESSION_LABEL,
    "pipeline_step": "4.2_extract_roi_timecourses",

    # Atlas & ROI definitions  (Point 1: canonical ROI)
    "atlas":        ATLAS_PARC,               # "aparc.a2009s"
    "roi_config":   ROI_CONFIG,               # full atlas citation, hemisphere rules,
                                              # aggregation mode, parcel rationale

    # Full label lists for Supplementary Materials:
    "vmpfc_labels_found":  [lb.name for lb in vmpfc_labels],
    "dlpfc_labels_found":  [lb.name for lb in dlpfc_labels],
    "vmpfc_n_labels":      len(vmpfc_labels),
    "dlpfc_n_labels":      len(dlpfc_labels),

    #  dSPM extraction
    "dspm_extraction_mode":   "mean",         # mne.extract_label_time_course mode
    "dspm_lambda2":           1.0 / 9.0,      # SNR=3 → λ²=1/9
    "dspm_method":            "MNE",
    "dspm_normalization":     "dSPM",
    "dspm_loose":             0.2,
    "dspm_depth":             0.8,
    "dspm_inv_method":        "Minimum Norm (SVD)",

    # LCMV parameters  (Point 2: fixed weight policy)
    # None if LCMV cells were not run
    "lcmv_run":     _lcmv_run,
    "lcmv_params":  _lcmv_params,

    #  Epoch metadata
    "n_trials_valid":  int(roi_tc_final.shape[0]),
    "n_timepoints":    int(len(epochs.times)),
    "tmin_s":          float(epochs.times[0]),
    "tmax_s":          float(epochs.times[-1]),
    "sfreq_hz":        float(epochs.info["sfreq"]),
    # Bandpass applied to epochs (from info):
    "epoch_highpass_hz": float(epochs.info.get("highpass", 0.0)),
    "epoch_lowpass_hz":  float(epochs.info.get("lowpass", 0.0)),

    #  Noise covariance
    "noise_cov_source": _lcmv_noise,    # "emptyroom" / "baseline" / "not_run"

    #  Leakage control  (Point 3: orth documentation)
    # None if LCMV-2 was not run
    "leakage_control": _leakage_qc,

    #  File names (Point 5: unambiguous per-subject/session)
    "output_files": {
        "dspm_npz":       str(OUTPUT_NPZ.name),
        "lcmv_npz":       f"roi_tc_LCMV_{SESSION_LABEL}.npz" if _lcmv_run else None,
        "leakage_fig":    f"leakage_control_{SESSION_LABEL}.png" if _leakage_run else None,
        "meta_json":      f"roi_tc_meta_{SESSION_LABEL}.json",
    },
}

meta_path = DERIV_OUTPUT / f"roi_tc_meta_{SESSION_LABEL}.json"
with open(meta_path, "w", encoding="utf-8") as _f:
    _json.dump(roi_meta, _f, indent=2, ensure_ascii=False)

print(f"✓ ROI metadata saved: {meta_path.name}")
print(f"\n  [Atlas & ROI]")
print(f"  atlas={ATLAS_PARC}, citation={ROI_CONFIG['atlas_citation'][:55]}...")
print(f"  hemisphere_rule={ROI_CONFIG['hemisphere_rule']}")
print(f"  vertex_aggregation_dspm={ROI_CONFIG['vertex_aggregation_dspm']}")
print(f"  vertex_aggregation_lcmv={ROI_CONFIG['vertex_aggregation_lcmv']}")

print(f"\n  [ROI labels for Supplementary]")
print(f"  vmPFC ({len(vmpfc_labels)} labels): {[lb.name for lb in vmpfc_labels]}")
print(f"  dlPFC ({len(dlpfc_labels)} labels): {[lb.name for lb in dlpfc_labels]}")

if _lcmv_run:
    print(f"\n  [LCMV params]")
    for k, v in _lcmv_params.items():
        print(f"    {k}: {v}")
else:
    print(f"\n  [LCMV] Not run — lcmv_params saved as null.")
    print(f"  Run LCMV-0 → LCMV-1 to populate LCMV fields.")

if _leakage_run:
    print(f"\n  [Leakage QC]")
    print(f"    corr_dspm_raw:  {_leakage_qc.get('corr_vmpfc_dlpfc_dspm_raw', 'n/a'):.3f}")
    print(f"    corr_dspm_orth: {_leakage_qc.get('corr_vmpfc_dlpfc_dspm_orth', 'n/a'):.3f}")
    print(f"    corr_lcmv:      {_leakage_qc.get('corr_vmpfc_dlpfc_lcmv', 'n/a'):.3f}")
    print(f"    sign_consistent: {_leakage_qc.get('effect_sign_consistent', 'n/a')}")
else:
    print(f"\n  [Leakage QC] Not run — leakage_control saved as null.")
    print(f"  Run LCMV-2 to populate leakage control fields.")


KeyboardInterrupt: 